# Лабораторная работа 4. Поиск подстроки

**Опора:** лекция 7.

Реализуются **три алгоритма поиска**: наивный, Кнута–Морриса–Пратта и
Рабина–Карпа. Задача у всех трёх одна.

- **Вход:** текст и образец.
- **Выход:** список позиций начала всех вхождений образца в текст, в порядке
  возрастания, включая перекрывающиеся вхождения. Если вхождений нет — пустой
  список.

```text
Вход:  текст = "ababa", образец = "aba"      Выход: [0, 2]
Вход:  текст = "abcdef", образец = "xyz"     Выход: []
```

Пустой образец считается входящим в каждую позицию, включая позицию за концом
текста: для текста длины `n` ответ — `[0, 1, …, n]`.

Тексты, образцы, измерение времени и графики уже готовы и
лежат в пакете `labkit` рядом с ноутбуком.

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "labkit").is_dir():
        sys.path.insert(0, str(candidate))
        break

import labkit as lk

print("Режим:", lk.CONFIG.name, "| длины текстов:", lk.text_sizes())

Режим: быстрый | длины текстов: [2000, 4000, 8000, 16000, 32000]


## Задача 1. Наивный поиск

Образец прикладывается к каждой позиции текста и сравнивается посимвольно.

In [6]:
def naive_search(text, pattern):
    """Список позиций всех вхождений pattern в text, включая перекрывающиеся."""

    n, m = len(text), len(pattern)
    report_match = []
    for i in range(n - m + 1): 
        j = 0
        while j < m and text[i + j] == pattern[j]:
            j += 1 
        if j == m: 
            report_match.append(i)
    return report_match
    raise NotImplementedError("naive_search")

In [7]:
lk.check_search(naive_search, "Наивный поиск")

Наивный поиск
  [ok]   пустой текст и образец
    Input: {'pattern': '', 'text': ''}
    Output: [0]
    Expected: [0]
  [ok]   пустой текст
    Input: {'pattern': 'a', 'text': ''}
    Output: []
    Expected: []
  [ok]   пустой образец
    Input: {'pattern': '', 'text': 'abc'}
    Output: [0, 1, 2, 3]
    Expected: [0, 1, 2, 3]
  [ok]   один символ найден
    Input: {'pattern': 'a', 'text': 'a'}
    Output: [0]
    Expected: [0]
  [ok]   один символ не найден
    Input: {'pattern': 'b', 'text': 'a'}
    Output: []
    Expected: []
  [ok]   в начале
    Input: {'pattern': 'abc', 'text': 'abcdef'}
    Output: [0]
    Expected: [0]
  [ok]   в середине
    Input: {'pattern': 'cde', 'text': 'abcdef'}
    Output: [2]
    Expected: [2]
  [ok]   в конце
    Input: {'pattern': 'def', 'text': 'abcdef'}
    Output: [3]
    Expected: [3]
  [ok]   образец длиннее текста
    Input: {'pattern': 'abcdef', 'text': 'abc'}
    Output: []
    Expected: []
  [ok]   несколько вхождений
    Input: {'pattern

## Задача 2. Алгоритм Кнута–Морриса–Пратта

Отдельно вычисляется префикс-функция образца: для каждого префикса — длина его
наибольшего собственного суффикса, который одновременно является его префиксом.
При несовпадении поиск использует префикс-функцию и не возвращается назад по
тексту.

```text
Вход префикс-функции:  "ababa"    Выход: [0, 0, 1, 2, 3]
```

In [28]:
def prefix_function(pattern):
    """Для каждого префикса — длина наибольшего собственного суффикса,
    который одновременно является его префиксом."""
    pi = [0] * len(pattern)
    for i in range(1, len(pattern)):
        j = pi[i - 1] 
        while j > 0 and pattern[i] != pattern[j]:
            j = pi[j - 1] 
        if pattern[i] == pattern[j]:
            j += 1 
        pi[i] = j 
    return pi
    raise NotImplementedError("prefix_function")


def kmp_search(text, pattern):
    """Список позиций всех вхождений pattern в text."""
    report_match = []
    m = len(pattern)
    if m == 0:
        return list(range(len(text) + 1))
    pi = prefix_function(pattern) 
    j = 0 
    for i in range(len(text)):
        while j > 0 and text[i] != pattern[j]:
            j = pi[j - 1]
        if text[i] == pattern[j]:
            j += 1 
        if j == m:
            report_match.append(i - m + 1) 
            j = pi[j - 1]
    return report_match 
    raise NotImplementedError("kmp_search")

In [29]:
lk.check_prefix_function(prefix_function)
lk.check_search(kmp_search, "Кнут–Моррис–Пратт")

Префикс-функция
  [ok]   префикс-функция
    Input: {'pattern': ''}
    Output: []
    Expected: []
  [ok]   префикс-функция
    Input: {'pattern': 'a'}
    Output: [0]
    Expected: [0]
  [ok]   префикс-функция
    Input: {'pattern': 'aaaa'}
    Output: [0, 1, 2, 3]
    Expected: [0, 1, 2, 3]
  [ok]   префикс-функция
    Input: {'pattern': 'abcd'}
    Output: [0, 0, 0, 0]
    Expected: [0, 0, 0, 0]
  [ok]   префикс-функция
    Input: {'pattern': 'ababa'}
    Output: [0, 0, 1, 2, 3]
    Expected: [0, 0, 1, 2, 3]
  [ok]   префикс-функция
    Input: {'pattern': 'aabaaab'}
    Output: [0, 1, 0, 1, 2, 2, 3]
    Expected: [0, 1, 0, 1, 2, 2, 3]
  [ok]   префикс-функция
    Input: {'pattern': 'abacabab'}
    Output: [0, 0, 1, 0, 1, 2, 3, 2]
    Expected: [0, 0, 1, 0, 1, 2, 3, 2]
  [ok]   200 случайных образцов
  Пройдено 8 из 8
Кнут–Моррис–Пратт
  [ok]   пустой текст и образец
    Input: {'pattern': '', 'text': ''}
    Output: [0]
    Expected: [0]
  [ok]   пустой текст
    Input: {'pattern':

## Задача 3. Алгоритм Рабина–Карпа

Сравниваются хеши образца и окна текста той же длины. Хеш здесь тот же
полиномиальный, что и у строковых ключей хеш-таблицы, — схема Горнера с
основанием `X` и модулем `q`:

```text
h = 0
для каждого символа c: h = (h * X + ord(c)) mod q
```

Отличие в том, что для каждого следующего окна хеш не вычисляется заново, а
пересчитывается за `O(1)` при сдвиге на один символ: из хеша вычитается вклад
ушедшего символа, остаток умножается на основание, прибавляется код пришедшего.
Такой хеш называется скользящим. Основание берётся не меньше размера алфавита,
модуль — большое простое число.

- **Требование:** при совпадении хешей обязательна проверка реального совпадения
  строк, поскольку возможны коллизии.

In [ ]:
X, q = 31, 13
xm = pow(X, m - 1, q)   
def roll(hs, out, inc):
    hs = (hs - ord(out) * xm) % q 
    hs = hs * X % q 
    hs = (hs + ord(inc)) % q 
    return hs
def rabin_karp_search(text, pattern):
    """Список позиций всех вхождений pattern в text."""
    report_match = []
    m = len(pattern)
    hp = hash_poly(pattern, q) 
    hs = hash_poly(text[:m], q)
    for i in range(len(text) - m + 1):
        if hs == hp: 
            if text[i:i + m] == P: 
                report_match.append(i)
        if i + m < len(text):
            hs = roll(hs, text[i], text[i + m])
    raise NotImplementedError("rabin_karp_search")

In [ ]:
lk.check_search(rabin_karp_search, "Рабин–Карп")

## Набор реализованных алгоритмов

Дальше всё считается по тем реализациям, которые действительно написаны. Все три
должны давать одинаковый ответ на одних и тех же данных.

In [ ]:
SEARCHES = lk.implemented_searches({
    "наивный": naive_search,
    "Кнут–Моррис–Пратт": kmp_search,
    "Рабин–Карп": rabin_karp_search,
})

print("В экспериментах участвуют:", ", ".join(SEARCHES))
lk.check_same_results(SEARCHES, lk.random_text(5_000), ["abc", "abcd", "zzz", "a"])

## Эксперимент 1. Размер входа

Слева растёт длина текста при образце длины 20, взятом из текста; справа
длина текста фиксирована, а растёт длина образца.

In [ ]:
by_size = lk.size_experiment(SEARCHES)
lk.plot_search_experiment(by_size, "Размер входа · время поиска");

## Эксперимент 2. Масштабирование алгоритмов

Длины текста `n` и образца `m` удваиваются одновременно, причём во всех точках
`m = n / 10`. Алгоритмы проверяются на случайном тексте, на входе, где
несовпадение находится примерно после ¾ образца, и на тексте с большим числом
вхождений.

Каждый столбец соответствует алгоритму. Сверху показано медианное время поиска,
снизу — то же время, делённое на суммарную длину входа `n + m`. Полоса вокруг
линии показывает интервал от первого до третьего квартиля.

In [ ]:
scaling = lk.scaling_experiment(SEARCHES)
lk.plot_scaling(scaling);

## Самостоятельные выводы

1. Почему увеличение длины текста и длины образца по-разному влияет на время
   поиска? Учтите число проверяемых окон `n − m + 1` и предварительную обработку
   образца.
2. Почему на случайном тексте наивный поиск может быть сопоставим с КМП и
   Рабином–Карпом, а на входе с длинным несовпадением начинает заметно
   проигрывать?
3. Как префикс-функция помогает КМП не возвращаться назад по тексту и находить
   перекрывающиеся вхождения?
4. Как скользящий хеш позволяет Рабину–Карпу обрабатывать следующее окно за
   `O(1)` и почему при совпадении хешей всё равно необходимо сравнивать строки?
5. Какие результаты графиков объясняются асимптотикой алгоритмов, а какие —
   устройством конкретных входов и постоянными затратами реализации на
   Python?